# Atelier Préparation de Données Tabulaires 
Contexte 
Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT. Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de fonctionnement et l'état du système de climatisation. Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning capable de prédire la consommation énergétique ou de détecter les situations anormales. Cependant, les données brutes présentent volontairement différents problèmes : valeurs manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables. L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt pour le Machine Learning. Objectifs pédagogiques À la fin de l'atelier, l'apprenant doit être capable de :
 1) explorer un jeu de données tabulaire ; 
 2) identifier les différents types de variables ; 
 3) détecter les problèmes de qualité ; 
 4) analyser les valeurs manquantes ; 
 5) détecter et traiter les doublons ; 
 6) identifier les valeurs aberrantes ; 
 7) détecter les incohérences ; 
 8) analyser le déséquilibre des classes ; 
 9) choisir une stratégie d'encodage adaptée ; 
 10) appliquer différents encodages catégoriels ; 
 11) normaliser et standardiser les variables numériques ; 
 12) éviter la fuite de données ; 
 13) construire un pipeline de preprocessing avec Scikit-learn ; 
 14) produire un dataset final exploitable par un algorithme de Machine Learning. 

# Partie 1 – Explorer les données 


## 1) Charger les données CSV ; 

In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("../data/smart_building_raw.csv")


## 2) Afficher les premières lignes du dataset ; 

In [4]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


## 3) Afficher les dernières lignes du dataset ; 


In [5]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


## 4) Combien d'observations contient le dataset ? 

In [6]:
print(len(df))
print(df.shape[0])

507
507


## 5) Combien de variables possède le dataset ? 


In [7]:
print(len(df.columns))
print(df.shape[1])


14
14


## 6) Identifier les variables numériques ; 

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_mesure           507 non-null    int64  
 1   date                507 non-null    str    
 2   batiment            507 non-null    str    
 3   type_batiment       503 non-null    str    
 4   zone                507 non-null    str    
 5   temperature         495 non-null    float64
 6   humidite            496 non-null    float64
 7   co2                 500 non-null    float64
 8   occupation          501 non-null    float64
 9   consommation_kwh    502 non-null    float64
 10  mode_climatisation  502 non-null    str    
 11  etat_systeme        507 non-null    str    
 12  jour_semaine        502 non-null    str    
 13  alerte              507 non-null    str    
dtypes: float64(5), int64(1), str(8)
memory usage: 55.6 KB


In [14]:
variables_numeriques = df.select_dtypes(include=[np.float64]).columns.tolist()

print(variables_numeriques)

['temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


## 7) Identifier les variables catégorielles

In [28]:
variables_categorielles = df.select_dtypes(include="str").columns.tolist()
variables_categorielles.remove("date")
print(variables_categorielles)

['batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


## 8) Identifier les dates 

In [29]:
dates = ["date"]
dates

['date']

## 9) Identifier les identifiants 

In [35]:
identifiants = df.select_dtypes(include=[np.int64]).columns.tolist()
identifiants

['id_mesure']

## 10) Déterminer les statistiques : moyenne, médiane, minimum, maximum, écart-type et quartiles. 

In [37]:
df[variables_numeriques].describe().round(2)

,temperature,humidite,co2,occupation,consommation_kwh
count,495.00,496.00,500.00,501.00,502.00
mean,24.15,57.86,844.15,44.85,169.07
std,7.42,16.03,582.18,24.95,53.16
min,-30.00,-12.00,89.00,-20.00,-100.00
25%,21.60,49.28,623.75,27.00,136.88
50%,24.00,57.55,787.50,46.00,169.80
75%,26.00,65.75,952.00,61.00,202.98
max,96.00,160.00,6000.00,116.00,336.20


## 11) Variables potentiellement problématiques

En comparant les valeurs min/max de `.describe()` aux limites physiquement possibles pour
chaque variable, plusieurs incohérences apparaissent :

- **temperature** : min = -30°C, max = 96°C → des valeurs impossibles pour un bâtiment
  climatisé (plage réaliste attendue : environ 15°C à 35°C).
- **humidite** : min = -12%, max = 160% → un taux d'humidité est un pourcentage, il ne
  peut donc pas être négatif ni dépasser 100%.
- **occupation** : min = -20 → un nombre de personnes ne peut pas être négatif.
- **consommation_kwh** : min = -100 → une consommation d'énergie ne peut pas être négative.
- **co2** : max = 6000 ppm → valeur élevée mais pas physiquement impossible ; à surveiller
  plutôt qu'à corriger d'office.




## 12) Pour les données incohérentes : 

### a) rechercher des valeurs telles que humidité < 0 ; 

In [49]:
print("a) humidité < 0   :", (df["humidite"] < 0).sum(), "lignes")


a) humidité < 0   : 3 lignes


In [ ]:
# acceder aux valeurs manquantes de la colonne humidité loc[ligne, colonne]

df.loc[df["humidite"] < 0, "humidite"]

281    -5.0
335    -8.0
366   -12.0
Name: humidite, dtype: float64

### b) rechercher des valeurs telles que humidité > 100 ; 

In [56]:
print("b) Valeurs humidité > 100 :")
print((df["humidite"] > 100).sum(), "lignes")
print(df.loc[df["humidite"] > 100, "humidite"].tolist())

b) Valeurs humidité > 100 :
7 lignes
[108.0, 145.0, 160.0, 125.0, 140.0, 132.0, 110.0]


### c) rechercher des valeurs telles que température extrêmement élevée ; 

In [61]:
print("c) Valeurs température extrême :")
print(((df["temperature"] > 45) ).sum(), "lignes")
print(df.loc[(df["temperature"] > 45) , "temperature"].tolist())

c) Valeurs température extrême :
4 lignes
[72.5, 96.0, 88.0, 95.2]


### d) rechercher des valeurs telles que occupation négative

In [63]:
print("d) Valeurs occupation négative :")
print((df["occupation"] < 0).sum(), "lignes")
print(df.loc[df["occupation"] < 0, "occupation"].tolist())

d) Valeurs occupation négative :
5 lignes
[-5.0, -8.0, -12.0, -2.0, -20.0]


### e) rechercher des valeurs telles que consommation négative ; 

In [64]:
print("e) Valeurs consommation négative :")
print((df["consommation_kwh"] < 0).sum(), "lignes")
print(df.loc[df["consommation_kwh"] < 0, "consommation_kwh"].tolist())

e) Valeurs consommation négative :
4 lignes
[-15.0, -50.0, -20.0, -100.0]


### f) Si une valeur est manifestement erronée et qu’on ne peut pas retrouver sa vraie valeur, la transformer en valeur manquante 



In [65]:
df.loc[df["humidite"] < 0, "humidite"] = np.nan
df.loc[df["humidite"] > 100, "humidite"] = np.nan
df.loc[(df["temperature"] > 45), "temperature"] = np.nan
df.loc[df["occupation"] < 0, "occupation"] = np.nan
df.loc[df["consommation_kwh"] < 0, "consommation_kwh"] = np.nan

### verification

In [66]:
print(df.loc[df["consommation_kwh"] < 0, "consommation_kwh"].tolist())

[]


### g) rechercher des valeurs telles que catégories mal orthographiées. 

In [ ]:
# lecture des valeurs uniques pour les variables catégorielles

for col in variables_categorielles:
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

--- batiment ---
<StringArray>
['B8', 'B7', 'B5', 'B6', 'B3', 'B4', 'B2', 'B1']
Length: 8, dtype: str

--- type_batiment ---
<StringArray>
[         'Entrepôt',            'Bureau', 'Centre commercial',
        'Université',           'Hôpital',             'École',
             'ÉCOLE',             'ecole',            'BUREAU',
                 nan,             'Bureu',            'bureau',
       ' UNIVERSITÉ',          'hôpital ', 'centre commercial',
          ' Bureau ',          'entrepot']
Length: 17, dtype: str

--- zone ---
<StringArray>
['A', 'D', 'B', 'C']
Length: 4, dtype: str

--- mode_climatisation ---
<StringArray>
['Eco', 'Normal', 'Boost', nan, 'normal', 'BOOST', 'normale', 'Normal ']
Length: 8, dtype: str

--- etat_systeme ---
<StringArray>
['Normal', 'Alerte', 'Panne']
Length: 3, dtype: str

--- jour_semaine ---
<StringArray>
['Jeudi', 'Lundi', 'Dimanche', 'Vendredi', 'Mercredi', 'Mardi', 'Samedi', nan]
Length: 8, dtype: str

--- alerte ---
<StringArray>
['Non', 'Oui

### 

**type_batiment** contient plusieurs variantes qui désignent la même catégorie :
- Variations de casse : "BUREAU", "Bureau", "bureau"
- Espaces parasites : " Bureau ", "hôpital "
- Accents manquants : "ecole" (école), "entrepot" (entrepôt)
- Faute de frappe : "Bureu" (bureau)

**mode_climatisation** présente le même type de problème :
- Casse : "BOOST", "Boost"
- Espace parasite : "Normal "
- Variante orthographique : "normale" (normal)

### h) normaliser les catégories textuelles en supprimant les espaces puis en uniformisant la casse

In [68]:
#Pour chaque valeur texte de cette colonne, enlève les espaces autour, 
# puis mets tout en minuscules, et remplace la colonne d'origine par le résultat.

for col in variables_categorielles:
    df[col] = df[col].str.strip().str.lower()

In [70]:
# Correction des valeurs pour les variables catégorielles faute de frappe ou d'orthographe

corrections_type_batiment = {
    "bureu": "bureau",
    "ecole": "école",
    "entrepot": "entrepôt",
}
df["type_batiment"] = df["type_batiment"].replace(corrections_type_batiment)

corrections_climatisation = {
    "normale": "normal",
}
df["mode_climatisation"] = df["mode_climatisation"].replace(corrections_climatisation)

In [77]:
# verification des corrections
for col in variables_categorielles:
    print(f"--- {col} ---")
    print(col, "→", sorted(df[col].dropna().unique()))  
    print()
    

--- batiment ---
batiment → ['b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8']

--- type_batiment ---
type_batiment → ['bureau', 'centre commercial', 'entrepôt', 'hôpital', 'université', 'école']

--- zone ---
zone → ['a', 'b', 'c', 'd']

--- mode_climatisation ---
mode_climatisation → ['boost', 'eco', 'normal']

--- etat_systeme ---
etat_systeme → ['alerte', 'normal', 'panne']

--- jour_semaine ---
jour_semaine → ['dimanche', 'jeudi', 'lundi', 'mardi', 'mercredi', 'samedi', 'vendredi']

--- alerte ---
alerte → ['non', 'oui']



## 13) Pour les valeurs manquantes : 

### a) Calculer le nombre et le pourcentage de valeurs manquantes par colonne 

In [79]:
manquants = df.isna().sum()
manquants 


id_mesure              0
date                   0
batiment               0
type_batiment          4
zone                   0
temperature           16
humidite              21
co2                    7
occupation            11
consommation_kwh       9
mode_climatisation     5
etat_systeme           0
jour_semaine           5
alerte                 0
dtype: int64

In [81]:
len(df)

507

In [80]:
pourcentage = (df.isna().sum() / len(df) * 100).round(2)
pourcentage



id_mesure             0.00
date                  0.00
batiment              0.00
type_batiment         0.79
zone                  0.00
temperature           3.16
humidite              4.14
co2                   1.38
occupation            2.17
consommation_kwh      1.78
mode_climatisation    0.99
etat_systeme          0.00
jour_semaine          0.99
alerte                0.00
dtype: float64

In [82]:
tableau_manquants = pd.DataFrame({"nb_manquants": manquants, "pct_manquants": pourcentage})
tableau_manquants = tableau_manquants[tableau_manquants["nb_manquants"] > 0].sort_values("nb_manquants", ascending=False)
print(tableau_manquants)

                    nb_manquants  pct_manquants
humidite                      21           4.14
temperature                   16           3.16
occupation                    11           2.17
consommation_kwh               9           1.78
co2                            7           1.38
mode_climatisation             5           0.99
jour_semaine                   5           0.99
type_batiment                  4           0.79


### b) Quelle variable possède le plus de valeurs manquantes ? 

humidite, avec 21 valeurs manquantes (4,14% du dataset).

### c) Quelle stratégie utiliser pour les valeurs manquantes ?

Chez nous, toutes les colonnes ont un taux de valeurs manquantes faible (< 5%). Dans ce cas,
plusieurs stratégies sont défendables :
- Imputation par la médiane pour les variables numériques (robuste aux valeurs extrêmes)
- Imputation par le mode pour les variables catégorielles


### d) Peut-on supprimer toutes les lignes contenant des valeurs manquantes ? 

Techniquement oui, mais ce n'est pas recommandé ici : en cumulant les colonnes concernées,
supprimer une ligne dès qu'UNE colonne est manquante ferait perdre bien plus que 4% des lignes
au total (les valeurs manquantes ne sont pas forcément sur les mêmes lignes d'une colonne à
l'autre). Sur seulement 507 observations, on ne peut pas se permettre de perdre autant de
données pour un projet de Machine Learning.